In [45]:
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect('churn_db.db')

In [ ]:
# Query1: Overall Churn Rate
query_1 = """
    SELECT
        COUNT(*) AS Total_Customers,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS Churned,
        ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100/ COUNT(*), 2) AS Churn_Rate_Percentage
    FROM customers;
"""

df_query1 = pd.read_sql_query(query1, conn)
print(df_query1)

   Total_Customers  Churned  Churn_Rate_Percentage
0             7043     2019                   28.0


In [ ]:
# Query 2: Churn by Contract (matches Excel)
query_2 = """
    SELECT
        contract,
        COUNT(*) AS Total,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS Churned,
        ROUND(SUM(CASE WHEN Churn='Yes' THEN 1 ELSE 0 END) * 100 / COUNT(*), 2) AS Churn_Rate_Percentage
    FROM customers
    GROUP BY Contract
    ORDER BY Churn_Rate_Percentage DESC;
"""

df_query2 = pd.read_sql_query(query2, conn)
print(df_query2)


  Contract  Total  Churned  Churn_Rate_Percentage
0       No    682      283                   41.0
1      Yes   6361     1736                   27.0


In [ ]:
# Query 3: Churn by Tenure (using CASE for binning)
query_3 = """
    SELECT
        CASE
            WHEN tenure <= 12 THEN '0-12 MONTHS'
            WHEN tenure <= 24 THEN '13-24 MONTHS'
            ELSE '25+ MONTHS'
            END AS Tenure_Group,
        COUNT(*) AS Total,
        ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Churn_Rate
    FROM customers
    GROUP BY Tenure_Group
    ORDER BY Churn_Rate DESC;
"""

df_query3 = pd.read_sql_query(query_3, conn)
print(df_query3)

  Tenure_Group  Total  Churn_Rate
0   25+ MONTHS   7043       28.67


In [ ]:
# Query 4: High-Risk Segment (multiple conditions)
query_4= """
    SELECT
        customerID,
        Contract,
        tenure,
        MonthlyCharges,
        Churn
    FROM customers
    WHERE 
    TRIM(LOWER(Contract)) = 'month-to-month'
    AND tenure <= 12
    AND CAST(MonthlyCharges AS DECIMAL(10,2)) > 70
    AND TRIM(LOWER(Churn)) = 'Yes';
"""

df_query4 = pd.read_sql_query(query_4, conn)
print(df_query4)


Empty DataFrame
Columns: [customerID, Contract, tenure, MonthlyCharges, Churn]
Index: []


In [ ]:
# Query 5: Find Current At-Risk Customers
query_5 = """
    
    SELECT
        customerID,
        Contract,
        tenure,
        MonthlyCharges,
        'At Risk' AS Risk_Flag
    FROM customers
    WHERE Contract = 'Month-to-Month'
      AND tenure <= 12
      AND MonthlyCharges > 70
      AND Churn = 'Yes';
"""
df_query5 = pd.read_sql_query(query_5, conn)
print(df_query5)

#Empty DataFrame seen because the filters are strict

Empty DataFrame
Columns: [customerID, Contract, tenure, MonthlyCharges, Risk_Flag]
Index: []
